# LLM-as-a-Judge · Experiment App
Run the single cell below. All configuration, execution, metrics and W&B logging are integrated.

**Dependencies:** `anywidget`, `ipywidgets` and the existing evaluation pipeline must already be installed.


In [ ]:

import anywidget
import traitlets
from importlib.resources import files
from pathlib import Path
from evaluation_pipeline.datasets.dataset_io import _get_project_root
from IPython.display import display


# ============================================
# LLM Judge Lab — Experiment Setup
# Judge Models with Provider Selection
# ============================================

def discover_prepared_datasets():
    from evaluation_pipeline.datasets.dataset_io import _get_project_root
    from evaluation_pipeline.datasets.config_loader import CONFIG_DIR, load_dataset_config

    prepared_dir = _get_project_root() / "datasets" / "prepared"
    if not prepared_dir.is_dir():
        return []

    names = sorted((p.stem for p in CONFIG_DIR.glob("*.yaml")), key=len, reverse=True)
    datasets = []
    for path in sorted(prepared_dir.glob("*.json")):
        for name in names:
            prefix = f"{name}_"
            if not path.stem.startswith(prefix):
                continue
            variant = path.stem[len(prefix):]
            if not variant:
                continue
            try:
                load_dataset_config(name)
            except (ValueError, FileNotFoundError):
                continue
            datasets.append({"name": name, "variant": variant, "dataset_id": path.stem})
            break
    return datasets


class ExperimentSetup(anywidget.AnyWidget):

    selected_backend = traitlets.Unicode(
        default_value="ollama"
    ).tag(sync=True)

    selected_models = traitlets.List(
        traitlets.Dict(),
        default_value=[]
    ).tag(sync=True)

    available_datasets = traitlets.List(traitlets.Dict(), default_value=[]).tag(sync=True)
    selected_datasets = traitlets.List(traitlets.Dict(), default_value=[]).tag(sync=True)
    selected_methods = traitlets.List(traitlets.Dict(), default_value=[]).tag(sync=True)
    confidence_mode = traitlets.Bool(default_value=False).tag(sync=True)
    use_wandb = traitlets.Bool(default_value=False).tag(sync=True)
    launch_request = traitlets.Int(default_value=0).tag(sync=True)
    stop_request = traitlets.Int(default_value=0).tag(sync=True)
    run_status = traitlets.Unicode(default_value="Ready").tag(sync=True)
    run_current = traitlets.Int(default_value=0).tag(sync=True)
    run_total = traitlets.Int(default_value=0).tag(sync=True)
    run_active = traitlets.Bool(default_value=False).tag(sync=True)
    current_run_label = traitlets.Unicode(default_value="").tag(sync=True)
    run_results = traitlets.List(traitlets.Dict(), default_value=[]).tag(sync=True)
    current_run_started_at = traitlets.Float(default_value=0.0).tag(sync=True)
    execution_busy = traitlets.Bool(default_value=False).tag(sync=True)
    sample_current = traitlets.Int(default_value=0).tag(sync=True)
    sample_total = traitlets.Int(default_value=0).tag(sync=True)
    sample_last_completed_at = traitlets.Float(default_value=0.0).tag(sync=True)

    _esm = files("evaluation_pipeline").joinpath("ui", "experiment.js").read_text(encoding="utf-8")
    _css = files("evaluation_pipeline").joinpath("ui", "experiment.css").read_text(encoding="utf-8")  


# ============================================
# Display Experiment Setup
# ============================================

experiment_ui = ExperimentSetup(available_datasets=discover_prepared_datasets())


# The interface is displayed only after the orchestrator is initialized.


def process_experiment_results(
    results, *, RUN_ID, JUDGE_MODEL, JUDGE_METHOD, DATASET_ID,
    BACKEND, EVALUATION_CRITERION, BASELINE_DETAIL,
    DYNAMIC_PREDICTION_VARIANT, CONFIDENCE_MODE,
):
    """Original notebook metrics and CSV export, isolated per run."""
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from pathlib import Path
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        cohen_kappa_score, matthews_corrcoef, confusion_matrix,
        ConfusionMatrixDisplay,
    )
    from evaluation_pipeline.prompts.registry import CRITERIA
    # ToDo: Move metrics code into evaluation_pipeline/metrics.py

    # ============================================
    # Prepare results
    # ============================================




    df_results = pd.DataFrame(results)

    total_samples = len(df_results)

    if total_samples == 0:
        raise ValueError(
            "No experiment results available."
        )


    # ============================================
    # Labels
    # ============================================

    # TODO:
    # Later load these labels directly from criterion YAML
    # to avoid duplicating label definitions in the notebook.



    label_config = CRITERIA[EVALUATION_CRITERION]["labels"]

    positive_label = label_config["positive"]
    negative_label = label_config["negative"]

    labels = [
        positive_label,
        negative_label,
    ]


    # ============================================
    # Validate dataset labels
    # ============================================

    invalid_true_labels = df_results[
        ~df_results["true_label"].isin(labels)
    ]

    if len(invalid_true_labels) > 0:
        raise ValueError(
            "Dataset contains unexpected true labels: "
            f"{sorted(invalid_true_labels['true_label'].unique())}"
        )


    # ============================================
    # Output quality
    # ============================================

    parsing_errors = df_results[
        df_results["predicted_label"]
        == "parsing_error"
    ]

    runtime_errors = df_results[
        df_results["predicted_label"]
        == "runtime_error"
    ]

    invalid_labels = df_results[
        (
            ~df_results["predicted_label"].isin(labels)
        )
        & (
            ~df_results["predicted_label"].isin(
                [
                    "parsing_error",
                    "runtime_error",
                ]
            )
        )
    ]

    valid_results = df_results[
        df_results["predicted_label"].isin(labels)
    ].copy()

    valid_samples = len(valid_results)
    invalid_samples = (
        total_samples - valid_samples
    )

    coverage = (
        valid_samples / total_samples
        if total_samples > 0
        else 0
    )

    print("Output Quality")
    print("=" * 40)
    print(f"Total samples: {total_samples}")
    print(
        f"Valid outputs: "
        f"{valid_samples} ({coverage:.2%})"
    )
    print(
        f"Parsing errors: "
        f"{len(parsing_errors)}"
    )
    print(
        f"Runtime errors: "
        f"{len(runtime_errors)}"
    )
    print(
        f"Invalid labels: "
        f"{len(invalid_labels)}"
    )


    # ============================================
    # Classification metrics
    # ============================================

    # Metrics calculated only on valid predictions.
    # If no valid predictions exist, these metrics
    # are unavailable rather than zero.

    accuracy = None
    precision = None
    recall = None
    f1 = None
    macro_f1 = None
    kappa = None
    mcc = None

    cm_labels = [negative_label, positive_label]
    cm = None

    if valid_samples > 0:

        y_true = valid_results["true_label"]
        y_pred = valid_results["predicted_label"]

        accuracy = float(accuracy_score(y_true, y_pred))

        precision = float(precision_score(
            y_true, y_pred,
            pos_label=positive_label,
            zero_division=0,
        ))

        recall = float(recall_score(
            y_true, y_pred,
            pos_label=positive_label,
            zero_division=0,
        ))

        f1 = float(f1_score(
            y_true, y_pred,
            pos_label=positive_label,
            zero_division=0,
        ))

        macro_f1 = float(f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0,
        ))

        kappa_value = float(cohen_kappa_score(y_true, y_pred))
        kappa = kappa_value if np.isfinite(kappa_value) else None

        mcc = float(matthews_corrcoef(y_true, y_pred))

        # ========================================
        # Confusion matrix
        # ========================================

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=cm_labels,
        )

        tn, fp, fn, tp = cm.ravel()

        print("\nClassification Metrics")
        print("=" * 40)

        print("Positive label:", positive_label)
        print("Negative label:", negative_label)

        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 (positive): {f1:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(
            f"Cohen's Kappa: {kappa:.4f}"
            if kappa is not None
            else "Cohen's Kappa: N/A"
        )
        print(f"MCC: {mcc:.4f}")
        print(f"Coverage: {coverage:.2%}")

        print("\nConfusion Matrix:")
        print(cm)

        print(
            f"TN={tn}, FP={fp}, "
            f"FN={fn}, TP={tp}"
        )

        # ========================================
        # Plot confusion matrix
        # ========================================

        fig, ax = plt.subplots(figsize=(7, 7))

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=cm_labels,
        )

        disp.plot(
            cmap=plt.cm.Blues,
            ax=ax,
            values_format="d",
        )

        for text in disp.text_.flatten():
            text.set_fontsize(24)

        ax.tick_params(
            axis="both",
            labelsize=16,
        )

        ax.set_xlabel(
            "Predicted label",
            fontsize=16,
        )

        ax.set_ylabel(
            "True label",
            fontsize=16,
        )

        ax.set_title(
            f"{EVALUATION_CRITERION} Confusion Matrix:\n"
            f"{RUN_ID} - {JUDGE_MODEL} - {JUDGE_METHOD}",
            fontsize=18,
            pad=15,
        )

        plt.tight_layout()
        plt.show()

    else:

        print("\nClassification Metrics")
        print("=" * 40)

        print("No valid predictions available.")
        print("Valid-only metrics: N/A")
        print(f"Coverage: {coverage:.2%}")
        print("Confusion matrix: N/A")

    # ============================================
    # Base summary metrics
    # ============================================

    summary_metrics = {
        "run_id": RUN_ID,
        "method": JUDGE_METHOD,
        "backend": BACKEND,
        "model": JUDGE_MODEL,
        "dataset_id": DATASET_ID,

        "evaluation_criterion": EVALUATION_CRITERION,
        "baseline_criterion_detail": BASELINE_DETAIL,

        "positive_label": positive_label,

        "total_samples": total_samples,
        "valid_samples": valid_samples,
        "invalid_samples": invalid_samples,
        "coverage": coverage,

        "parsing_errors": len(parsing_errors),
        "runtime_errors": len(runtime_errors),
        "invalid_labels": len(invalid_labels),

        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": macro_f1,
        "cohen_kappa": kappa,
        "mcc": mcc,
    }

    summary_metrics["valid_output_accuracy"] = accuracy

    summary_metrics["end_to_end_accuracy"] = float(
        (
            df_results["predicted_label"]
            == df_results["true_label"]
        ).mean()
    )

    # For Baseline and Dynamic, report accuracy over the full dataset.
    # Invalid final outputs count as unsuccessful evaluations.
    # The previous valid-only accuracy remains in valid_output_accuracy.
    if JUDGE_METHOD in {"baseline", "dynamic"}:
        summary_metrics["accuracy"] = summary_metrics["end_to_end_accuracy"]

    # ============================================
    # Second-Level analysis
    # ============================================

    second_level_metrics = {}

    if JUDGE_METHOD == "second_level":

        # ============================================
        # Valid First-Level outputs
        # ============================================
        # Only examples with a valid First-Level
        # label can proceed to Second-Level.
        # Valid does not mean correct.

        valid_first_level = df_results[
            df_results["first_level_label"].isin(labels)
        ].copy()

        valid_first_level_samples = len(valid_first_level)

        first_level_coverage = (
            valid_first_level_samples / total_samples
            if total_samples else 0
        )

        # ----------------------------------------
        # Accuracy before and after Second-Level
        # ----------------------------------------
        # Both accuracies use the same samples.
        # Invalid Second-Level outputs count
        # as incorrect final predictions.

        first_correct = valid_first_level[
            "first_level_label"
        ].eq(valid_first_level["true_label"])

        final_correct = valid_first_level[
            "predicted_label"
        ].eq(valid_first_level["true_label"])

        first_level_accuracy = (
            float(first_correct.mean())
            if valid_first_level_samples else None
        )

        final_accuracy = (
            float(final_correct.mean())
            if valid_first_level_samples else None
        )

        accuracy_delta = (
            final_accuracy - first_level_accuracy
            if valid_first_level_samples else None
        )

         # ----------------------------------------
        # Classification effects
        # ----------------------------------------

        corrected = (~first_correct) & final_correct
        degraded = first_correct & (~final_correct)

        unchanged_correct = first_correct & final_correct
        unchanged_wrong = (~first_correct) & (~final_correct)

        corrected_count = int(corrected.sum())
        degraded_count = int(degraded.sum())

        net_gain_count = corrected_count - degraded_count

        normalized_net_gain = (
            net_gain_count / valid_first_level_samples
            if valid_first_level_samples else None
        )

        # ----------------------------------------
        # Correction and degradation rates
        # ----------------------------------------

        n_first_wrong = int((~first_correct).sum())
        n_first_correct = int(first_correct.sum())

        correction_rate = (
            corrected_count / n_first_wrong
            if n_first_wrong else None
        )

        degradation_rate = (
            degraded_count / n_first_correct
            if n_first_correct else None
        )

        # ----------------------------------------
        # Valid classification changes
        # ----------------------------------------
        # Count label changes only when the final
        # prediction is a valid classification.

        valid_classification = valid_first_level[
            valid_first_level["predicted_label"].isin(labels)
        ].copy()

        label_changed = valid_classification[
            "predicted_label"
        ].ne(valid_classification["first_level_label"])

        label_change_count = int(label_changed.sum())

        label_change_rate = (
            float(label_changed.mean())
            if len(valid_classification) else None
        )

        # ----------------------------------------
        # Second-Level coverage
        # ----------------------------------------

        second_level_attempted = int(
            valid_first_level["second_level_prompt"]
            .notna()
            .sum()
        )

        valid_second_level = int(
            valid_first_level["second_level_verdict"]
            .isin(["correct", "not_correct"])
            .sum()
        )

        second_level_coverage = (
            valid_second_level / total_samples
            if total_samples else 0
        )

        classification_coverage = (
            len(valid_classification) / total_samples
            if total_samples else 0
        )

         # ----------------------------------------
        # Second-Level verdict inconsistency
        # ----------------------------------------
        # The judge declares the first verdict incorrect,
        # but proposes exactly the same classification label.
        # This is diagnostic only; final labels and accuracy stay unchanged.

        verdict_inconsistency_count = int((
            valid_first_level["second_level_verdict"].eq("not_correct")
            & valid_first_level["predicted_label"].eq(
                valid_first_level["first_level_label"]
            )
        ).sum())

        # ----------------------------------------
        # Store metrics
        # ----------------------------------------

        second_level_metrics = {
            "valid_first_level_samples": valid_first_level_samples,
            "first_level_coverage": first_level_coverage,
            "second_level_attempted": second_level_attempted,

            "first_level_accuracy": first_level_accuracy,
            "final_accuracy": final_accuracy,
            "accuracy_delta": accuracy_delta,

            "corrected_count": corrected_count,
            "degraded_count": degraded_count,
            "verdict_inconsistency_count": verdict_inconsistency_count,
            "net_gain_count": net_gain_count,
            "normalized_net_gain": normalized_net_gain,

            "correction_rate": correction_rate,
            "degradation_rate": degradation_rate,

            "label_change_count": label_change_count,
            "label_change_rate": label_change_rate,

            "second_level_coverage": second_level_coverage,
            "classification_coverage": classification_coverage,
        }

        # Primary Second-Level accuracy:
        # all valid First-Level samples are counted,
        # including failed Second-Level predictions.

        summary_metrics["accuracy"] = final_accuracy

        # ----------------------------------------
        # Print results
        # ----------------------------------------

        print("\nSecond-Level Analysis")
        print("=" * 40)

        print(
            f"Valid First-Level samples: "
            f"{valid_first_level_samples}/{total_samples} "
            f"({first_level_coverage:.2%})"
        )

        print(
            f"Second-Level calls attempted: "
            f"{second_level_attempted}"
        )

        if valid_first_level_samples:

            print(
                f"First-Level accuracy: "
                f"{first_level_accuracy:.2%}"
            )

            print(
                f"Final accuracy: "
                f"{final_accuracy:.2%}"
            )

            print(
                f"Accuracy delta: "
                f"{accuracy_delta:+.2%}"
            )

            print("\nClassification effects:")

            print(f"Corrected: {corrected_count}")
            print(f"Degraded: {degraded_count}")
            print(f"Verdict inconsistencies: {verdict_inconsistency_count}")

            print(
                f"Unchanged correct: "
                f"{int(unchanged_correct.sum())}"
            )

            print(
                f"Unchanged wrong: "
                f"{int(unchanged_wrong.sum())}"
            )

            print(f"Net gain: {net_gain_count}")

            print(
                f"Normalized net gain: "
                f"{normalized_net_gain:+.2%}"
            )

            print("\nRates:")

            print(
                f"Correction rate: {correction_rate:.2%}"
                if correction_rate is not None
                else "Correction rate: N/A"
            )

            print(
                f"Degradation rate: {degradation_rate:.2%}"
                if degradation_rate is not None
                else "Degradation rate: N/A"
            )

        else:
            print(
                "No valid First-Level outputs; "
                "Second-Level metrics unavailable."
            )


    # ============================================
    # Dynamic analysis
    # ============================================

    dynamic_metrics = {}

    if JUDGE_METHOD == "dynamic":

        if "prediction_raw_output" in df_results.columns:

            has_prediction = (
                df_results[
                    "prediction_raw_output"
                ].notna()
                & (
                    df_results[
                        "prediction_raw_output"
                    ]
                    .astype(str)
                    .str.strip()
                    != ""
                )
            )

        else:
            has_prediction = pd.Series(
                False,
                index=df_results.index,
            )

        prediction_available_count = int(
            has_prediction.sum()
        )

        prediction_output_coverage = (
            has_prediction.mean()
            if len(df_results) > 0
            else 0
        )

        dynamic_metrics = {
            "prediction_available_count": (
                prediction_available_count
            ),
            "prediction_coverage": (
                prediction_output_coverage
            ),
        }

        print("\nDynamic Analysis")
        print("=" * 40)

        print(
            f"Prediction available: "
            f"{prediction_available_count}"
        )

        print(
            f"Prediction coverage: "
            f"{prediction_output_coverage:.2%}"
        )


    # ============================================
    # Primary accuracy summary
    # ============================================

    print("\nPrimary Accuracy Summary")
    print("=" * 40)

    if JUDGE_METHOD == "second_level":
        denominator = second_level_metrics["valid_first_level_samples"]
        denominator_name = "valid First-Level samples"
    else:
        denominator = total_samples
        denominator_name = "all dataset samples"

    primary_accuracy = summary_metrics["accuracy"]

    print(f"Method: {JUDGE_METHOD}")
    print(f"Denominator: {denominator} ({denominator_name})")

    if primary_accuracy is not None:
        print(f"Primary accuracy: {primary_accuracy:.2%}")
    else:
        print("Primary accuracy: N/A")

    valid_accuracy = summary_metrics["valid_output_accuracy"]

    if valid_accuracy is not None:
        print(f"Valid-output accuracy: {valid_accuracy:.2%}")
    else:
        print("Valid-output accuracy: N/A")

    print(f"Output coverage: {coverage:.2%}")


    # ============================================
    # Confidence-Verteilung in summary_metrics
    # ============================================

    if "first_level_confidence" in df_results.columns:
        conf = pd.to_numeric(
            df_results["first_level_confidence"], errors="coerce"
        )
        conf_counts = conf.value_counts(normalize=True).sort_index()

        for level in [1, 2, 3, 4, 5]:
            summary_metrics[f"confidence_{level}_pct"] = round(
                float(conf_counts.get(level, 0.0)) * 100, 1
            )

        summary_metrics["confidence_mean"] = round(float(conf.mean()), 2)
        summary_metrics["confidence_valid_count"] = int(conf.notna().sum())
        summary_metrics["confidence_missing_count"] = int(conf.isna().sum())
    
    # ============================================
    # Resource usage (all executed model calls)
    # ============================================

    usage_fields = [
        "input_tokens", "output_tokens", "total_tokens", "latency_seconds"
    ]

    for field in usage_fields:
        if field not in df_results.columns:
            df_results[field] = float("nan")
        df_results[field] = pd.to_numeric(df_results[field], errors="coerce")

    # Only count complete token records; avoid treating missing data as zero.
    token_complete = df_results[
        ["input_tokens", "output_tokens", "total_tokens"]
    ].notna().all(axis=1)

    summary_metrics["token_usage_valid_count"] = int(token_complete.sum())
    summary_metrics["token_usage_missing_count"] = int((~token_complete).sum())
    summary_metrics["token_usage_coverage"] = (
        float(token_complete.mean())
    )

    for field in ["input_tokens", "output_tokens", "total_tokens"]:
        summary_metrics[f"avg_{field}"] = (
            float(df_results.loc[token_complete, field].mean())
            if token_complete.any() else None
        )

    summary_metrics["total_tokens_run"] = (
        int(df_results.loc[token_complete, "total_tokens"].sum())
        if token_complete.any() else None
    )

    summary_metrics["avg_latency_seconds"] = (
        float(df_results["latency_seconds"].mean())
        if df_results["latency_seconds"].notna().any() else None
    )

    print("\nResource Usage")
    for field in [
        "avg_input_tokens", "avg_output_tokens", "avg_total_tokens",
        "avg_latency_seconds", "total_tokens_run",
        "token_usage_valid_count", "token_usage_missing_count",
        "token_usage_coverage"
    ]:
        print(f"{field}: {summary_metrics[field]}")


    # ============================================
    # Estimated API cost (OpenRouter)
    # ============================================

    # USD per 1 million tokens.
    # Update these rates when provider pricing changes.
    MODEL_PRICING_USD_PER_M = {
        "openai/gpt-4.1-nano": {
            "input": 0.10,
            "output": 0.40,
        },
    }

    pricing = (
        MODEL_PRICING_USD_PER_M.get(JUDGE_MODEL)
        if BACKEND == "openrouter"
        else None
    )

    # Missing token usage must not be interpreted as zero cost.
    df_results["estimated_cost_usd"] = float("nan")

    if pricing is not None:
        df_results.loc[token_complete, "estimated_cost_usd"] = (
            df_results.loc[token_complete, "input_tokens"]
            * pricing["input"]
            + df_results.loc[token_complete, "output_tokens"]
            * pricing["output"]
        ) / 1_000_000

    cost_valid = df_results["estimated_cost_usd"].notna()

    summary_metrics["cost_coverage"] = float(cost_valid.mean())
    summary_metrics["cost_valid_count"] = int(cost_valid.sum())

    summary_metrics["avg_estimated_cost_usd"] = (
        float(df_results.loc[cost_valid, "estimated_cost_usd"].mean())
        if cost_valid.any()
        else None
    )

    summary_metrics["total_estimated_cost_usd"] = (
        float(df_results.loc[cost_valid, "estimated_cost_usd"].sum())
        if cost_valid.any()
        else None
    )

    summary_metrics["cost_currency"] = "USD"
    summary_metrics["cost_pricing_model"] = (
        JUDGE_MODEL if pricing is not None else None
    )

    print("\nEstimated API Cost")
    print("=" * 40)
    print("Pricing available:", pricing is not None)
    print("Cost coverage:", f"{summary_metrics['cost_coverage']:.2%}")
    print(
        "Average estimated cost (USD):",
        summary_metrics["avg_estimated_cost_usd"],
    )
    print(
        "Total estimated cost (USD):",
        summary_metrics["total_estimated_cost_usd"],
    )

    # ============================================
    # Final summary
    # ============================================

    summary_metrics.update(
        second_level_metrics
    )

    summary_metrics.update(
        dynamic_metrics
    )

    df_summary_metrics = pd.DataFrame(
        [summary_metrics]
    )


    # ============================================
    # Save CSV files
    # ============================================

    RESULTS_DIR = Path("../results")

    RESULTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_model = (
        JUDGE_MODEL
        .replace(":", "_")
        .replace("/", "_")
    )

    base_name = (
        f"{RUN_ID}_"
        f"{BACKEND}_"
        f"{JUDGE_METHOD}_"
        f"{safe_model}_"
        f"{DATASET_ID}"
    )

    results_path = (
        RESULTS_DIR
        / f"{base_name}_results.csv"
    )

    summary_path = (
        RESULTS_DIR
        / f"{base_name}_summary.csv"
    )

    df_results.to_csv(
        results_path,
        index=False,
    )

    df_summary_metrics.to_csv(
        summary_path,
        index=False,
    )

    print("\nSaved experiment results:")
    print(results_path)
    print(summary_path)
    return {
        "df_results": df_results,
        "df_summary_metrics": df_summary_metrics,
        "summary_metrics": summary_metrics,
        "positive_label": positive_label,
        "valid_samples": valid_samples,
        "fig": locals().get("fig"),
        "confusion_matrix": (locals()["cm"].tolist() if "cm" in locals() and hasattr(locals()["cm"], "tolist") else None),
        "results_path": results_path,
        "summary_path": summary_path,
    }


def log_experiment_wandb(
    processed, *, RUN_ID, JUDGE_MODEL, JUDGE_METHOD, DATASET_ID,
    BACKEND, EVALUATION_CRITERION, BASELINE_DETAIL,
    DYNAMIC_PREDICTION_VARIANT, CONFIDENCE_MODE,
    USE_WANDB, WANDB_PROJECT_NAME,
):
    """Original notebook W&B logic, isolated per run."""
    df_results = processed["df_results"]
    df_summary_metrics = processed["df_summary_metrics"]
    summary_metrics = processed["summary_metrics"]
    positive_label = processed["positive_label"]
    valid_samples = processed["valid_samples"]
    fig = processed["fig"]
    # ============================================
    # Weights & Biases logging
    # ============================================



    VARIANT_SHORT = {
        "original": "orig",
        "minimal": "min",
        "basic": "basic",
        "detailed": "det",
        "extended": "ext",
        "paper_based": "paper",
    }


    def short_variant(value):
        return VARIANT_SHORT.get(value, value)


    method_parts = [JUDGE_METHOD, short_variant(BASELINE_DETAIL)]
    if JUDGE_METHOD == "dynamic":
        method_parts.append(short_variant(DYNAMIC_PREDICTION_VARIANT))

    confidence_name = "skala" if CONFIDENCE_MODE else "no_skala"
    WANDB_RUN_NAME = (
        f"{BACKEND}_{JUDGE_MODEL}_{'-'.join(method_parts)}_"
        f"{DATASET_ID}_{confidence_name}({RUN_ID})"
    )

    # Scalar metrics available for all methods.
    COMMON_METRICS = (
        "accuracy",
        "valid_output_accuracy",
        "end_to_end_accuracy",
        "precision",
        "recall",
        "f1",
        "macro_f1",
        "cohen_kappa",
        "mcc",
        "coverage",
        "valid_samples",
        "invalid_samples",
    )

    SECOND_LEVEL_METRICS = (
        "valid_first_level_samples",
        "first_level_coverage",
        "second_level_attempted",
        "first_level_accuracy",
        "final_accuracy",
        "accuracy_delta",
        "corrected_count",
        "degraded_count",
        "verdict_inconsistency_count",
        "net_gain_count",
        "normalized_net_gain",
        "correction_rate",
        "degradation_rate",
        "label_change_count",
        "label_change_rate",
        "second_level_coverage",
        "classification_coverage",
    )

    DYNAMIC_METRICS = (
        "prediction_coverage",
        "prediction_available_count",
    )
    RESOURCE_METRICS = (
        "avg_input_tokens",
        "avg_output_tokens",
        "avg_total_tokens",
        "avg_latency_seconds",
        "total_tokens_run",
        "token_usage_valid_count",
        "token_usage_missing_count",
        "token_usage_coverage",
        "avg_estimated_cost_usd",
        "total_estimated_cost_usd",
        "cost_coverage",
    )
    CONFIDENCE_METRICS = (
        "confidence_mean",
        "confidence_valid_count",
        "confidence_missing_count",
        "confidence_1_pct",
        "confidence_2_pct",
        "confidence_3_pct",
        "confidence_4_pct",
        "confidence_5_pct",
    )

    metric_keys = list(COMMON_METRICS) + list(RESOURCE_METRICS) + list(CONFIDENCE_METRICS)
    if JUDGE_METHOD == "second_level":
        metric_keys.extend(SECOND_LEVEL_METRICS)
        # Verify that the new metric was calculated.
        assert "verdict_inconsistency_count" in summary_metrics, (
            "verdict_inconsistency_count is missing from summary_metrics"
        )

        print(
            "Verdict inconsistencies:",
            summary_metrics["verdict_inconsistency_count"]
        )
    elif JUDGE_METHOD == "dynamic":
        metric_keys.extend(DYNAMIC_METRICS)

    # None means unavailable, not zero. Missing metrics are not logged.
    metrics_to_log = {
        key: summary_metrics[key]
        for key in metric_keys
        if key in summary_metrics and summary_metrics[key] is not None
    }

    # Prepare per-example results without modifying the original DataFrame.
    df_wandb_results = df_results.copy()
    df_wandb_results["backend"] = BACKEND
    df_wandb_results["dataset_id"] = DATASET_ID

    result_columns = [
        "id",
        "dataset_id", 
        "backend",
        "true_label",
        "first_prompt",
        "first_raw_output",
        "first_level_label",
        "first_level_explanation",
        "confidence_mode",
        "first_level_confidence",
        "first_level_confidence_invalid",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "latency_seconds",
        "predicted_label",
    ]

    if JUDGE_METHOD == "second_level":
        result_columns.extend([
            "second_level_prompt",
            "second_level_raw_output",
            "second_level_verdict",
            "second_level_explanation",
            "second_level_corrected_verdict",
            "second_level_corrected_explanation",
        ])
    elif JUDGE_METHOD == "dynamic":
        df_wandb_results["dynamic_prediction_variant"] = DYNAMIC_PREDICTION_VARIANT
        result_columns.extend([
            "prediction_prompt",
            "prediction_raw_output",
            "dynamic_prediction_variant",
        ])

    # OpenRouter metadata for each model call
    OPENROUTER_STAGES = (
        "prediction",
        "first_level",
        "second_level",
    )

    OPENROUTER_FIELDS = (
        "openrouter_generation_id",
        "openrouter_response_model",
        "openrouter_provider",
    )

    result_columns.extend(
        f"{stage}_{field}"
        for stage in OPENROUTER_STAGES
        for field in OPENROUTER_FIELDS
    )

    df_wandb_results = df_wandb_results[
        [column for column in result_columns if column in df_wandb_results.columns]
    ].copy()

    wandb_url = None
    if not USE_WANDB:
        print("W&B logging disabled.")
    else:
        try:
            import wandb
        except ImportError as exc:
            raise ImportError("W&B logging requested but wandb is not installed.") from exc
        else:
            run = None
            try:
                run = wandb.init(
                    project=WANDB_PROJECT_NAME,
                    name=WANDB_RUN_NAME,
                    config={
                        "run_id": RUN_ID,
                        "method": JUDGE_METHOD,
                        "model": JUDGE_MODEL,
                        "backend": BACKEND,
                        "dataset_id": DATASET_ID,
                        "confidence_mode": CONFIDENCE_MODE,
                        "evaluation_criterion": EVALUATION_CRITERION,
                        "baseline_criterion_detail": BASELINE_DETAIL,
                        "positive_label": positive_label,
                        "temperature": 0,
                        "top_p": None,
                        "seed": None,
                        "json_mode_final": True,
                        "json_mode_prediction": (
                            False if JUDGE_METHOD == "dynamic" else None
                        ),
                        "dynamic_prediction_variant": (
                            DYNAMIC_PREDICTION_VARIANT
                            if JUDGE_METHOD == "dynamic"
                            else ""
                        ),
                    },
                    reinit="finish_previous",
                )

                if run is None:
                    raise RuntimeError("wandb.init() returned no run")

                run.log(metrics_to_log)

                wandb_tables = {
                    "results": wandb.Table(dataframe=df_wandb_results),
                    "summary_metrics": wandb.Table(dataframe=df_summary_metrics),
                }

                if valid_samples > 0:
                    wandb_tables["confusion_matrix"] = wandb.Image(fig)

                wandb.log(wandb_tables)

                wandb_url = run.url if run is not None else None
                print(f"W&B logging complete: {WANDB_RUN_NAME}")

            except Exception as exc:
                print(f"W&B logging failed: {type(exc).__name__}: {exc}")
                raise
            finally:
                if run is not None:
                    wandb.finish()
    return wandb_url

from itertools import product
from datetime import datetime
from uuid import uuid4
from evaluation_pipeline.datasets.dataset_io import load_prepared_dataset
from evaluation_pipeline.datasets.config_loader import load_dataset_config
from evaluation_pipeline.runner import run_judge_experiment

WANDB_PROJECT_NAME = "Experiments_LLM_as_a_judge"

def build_plan(ui):
    models = list(ui.selected_models)
    datasets = list(ui.selected_datasets)
    methods = list(ui.selected_methods)
    backend = ui.selected_backend
    if backend not in {"ollama", "openrouter"}:
        raise ValueError(f"Unsupported backend: {backend}")
    if not all((models, datasets, methods)):
        raise ValueError("Select at least one model, dataset and method.")
    if any(m["backend"] != backend for m in models):
        raise ValueError("Selected models must match the active backend.")
    available = {d["dataset_id"] for d in ui.available_datasets}
    if any(d["dataset_id"] not in available for d in datasets):
        raise ValueError("Selected dataset is no longer available.")

    plan = []
    for m, d, method in product(models, datasets, methods):
        method_name = method["method"]
        if method_name not in {"baseline", "second_level", "dynamic"}:
            raise ValueError(f"Unknown method: {method_name}")
        detail = method["baseline_criterion_detail"]
        if detail not in {"basic", "detailed"}:
            raise ValueError(f"Unknown criterion detail: {detail}")
        prediction = (
            method["dynamic_prediction_variant"] if method_name == "dynamic"
            else "original"
        )
        if method_name == "dynamic" and prediction not in {"original", "paper_based"}:
            raise ValueError(f"Unknown dynamic variant: {prediction}")
        dataset_config = load_dataset_config(d["name"])
        plan.append({
            "backend": backend,
            "model": m["model"],
            "dataset_id": d["dataset_id"],
            "dataset_name": d["name"],
            "method": method_name,
            "evaluation_criterion": dataset_config["evaluation_criterion"],
            "baseline_criterion_detail": detail,
            "dynamic_prediction_variant": prediction,
            "confidence_mode": bool(ui.confidence_mode),
            "use_wandb": bool(ui.use_wandb),
        })
    return plan


# Thread-local progress reporting around runner.tqdm; runner.py remains unchanged.
import evaluation_pipeline.runner as _judge_runner_module
import threading as _sample_threading
_sample_progress_local = _sample_threading.local()
if not getattr(_judge_runner_module.tqdm, "_experiment_progress_wrapper", False):
    _original_runner_tqdm = _judge_runner_module.tqdm
    def _notebook_progress_tqdm(iterable, *args, **kwargs):
        callback = getattr(_sample_progress_local, "callback", None)
        if callback is None:
            return _original_runner_tqdm(iterable, *args, **kwargs)
        def _tracked():
            total = len(iterable) if hasattr(iterable, "__len__") else None
            callback(0, total)
            completed = 0
            with _original_runner_tqdm(iterable, *args, **kwargs) as bar:
                for example in bar:
                    yield example
                    completed += 1
                    callback(completed, total)
        return _tracked()
    _notebook_progress_tqdm._experiment_progress_wrapper = True
    _judge_runner_module.tqdm = _notebook_progress_tqdm

def execute_plan(plan, progress_callback=None, stop_requested=None, on_run_start=None, on_run_result=None, on_sample_progress=None):
    """Run each configuration through the EXISTING judge, metrics and W&B code."""
    completed = []
    dataset_cache = {}
    if progress_callback is not None:
        progress_callback(0, len(plan), None)
    for index, item in enumerate(plan, 1):
        if stop_requested is not None and stop_requested():
            print(f"Stop requested: skipping {len(plan) - len(completed)} remaining run(s).")
            break
        dataset_id = item["dataset_id"]
        if dataset_id not in dataset_cache:
            dataset_cache[dataset_id] = load_prepared_dataset(dataset_id)
        run_id = datetime.now().strftime("r_%m%d_%H%M%S") + "_" + uuid4().hex[:8]
        print(f"\\n[{index}/{len(plan)}] {item['model']} | {dataset_id} | {item['method']}")
        if on_run_start is not None:
            on_run_start(index, len(plan), item)
        _sample_progress_local.callback = on_sample_progress
        try:
            results = run_judge_experiment(
                dataset=dataset_cache[dataset_id],
                run_id=run_id,
                model=item["model"],
                method=item["method"],
                evaluation_criterion=item["evaluation_criterion"],
                baseline_criterion_detail=item["baseline_criterion_detail"],
                dataset_id=dataset_id,
                dynamic_prediction_variant=item["dynamic_prediction_variant"],
                confidence_mode=item["confidence_mode"],
                backend=item["backend"],
            )
        finally:
            _sample_progress_local.callback = None
        args = dict(
            RUN_ID=run_id, JUDGE_MODEL=item["model"], JUDGE_METHOD=item["method"],
            DATASET_ID=dataset_id, BACKEND=item["backend"],
            EVALUATION_CRITERION=item["evaluation_criterion"],
            BASELINE_DETAIL=item["baseline_criterion_detail"],
            DYNAMIC_PREDICTION_VARIANT=item["dynamic_prediction_variant"],
            CONFIDENCE_MODE=item["confidence_mode"],
        )
        processed = process_experiment_results(results, **args)
        # The experiment is complete once its CSV files have been saved.
        # W&B is optional and must not prevent the next model from running.
        wandb_status = "disabled"
        wandb_url = None
        if item["use_wandb"]:
            try:
                wandb_url = log_experiment_wandb(
                    processed, **args, USE_WANDB=True,
                    WANDB_PROJECT_NAME=WANDB_PROJECT_NAME,
                )
                wandb_status = "logged"
            except Exception as exc:
                wandb_status = f"failed: {type(exc).__name__}: {exc}"
                print(f"W&B warning (CSV already saved): {wandb_status}")

        metrics = processed["summary_metrics"]
        cm = processed["confusion_matrix"]
        result_record = {
            "model": item["model"], "dataset_id": dataset_id,
            "method": item["method"],
            "baseline_detail": item["baseline_criterion_detail"],
            "prediction_variant": item["dynamic_prediction_variant"],
            "accuracy": float(metrics["accuracy"]) if metrics.get("accuracy") is not None else None,
            "macro_f1": float(metrics["macro_f1"]) if metrics.get("macro_f1") is not None else None,
            "coverage": float(metrics["coverage"]) if metrics.get("coverage") is not None else None,
            "valid_samples": int(metrics.get("valid_samples", 0)),
            "total_samples": int(metrics.get("total_samples", 0)),
            "runtime_errors": int(metrics.get("runtime_errors", 0)),
            "confusion_matrix": cm,
            "wandb_url": wandb_url,
            "wandb_status": wandb_status,
            "run_id": run_id,
        }
        if on_run_result is not None:
            on_run_result(result_record)
        completed.append({
            "configuration": item,
            "run_id": run_id,
            "summary_path": str(processed["summary_path"]),
            "results_path": str(processed["results_path"]),
            "wandb_status": wandb_status,
        })
        if processed["fig"] is not None:
            import matplotlib.pyplot as plt
            plt.close(processed["fig"])
        if progress_callback is not None:
            progress_callback(index, len(plan), item)
    return completed


from IPython.display import display, clear_output
import ipywidgets as widgets
import threading
import builtins
import time

# Kernel-scoped state survives re-running this notebook cell.
# It also detects workers started by older versions of this notebook.
if not hasattr(builtins, "_llm_judge_execution_guard"):
    builtins._llm_judge_execution_guard = {
        "lock": threading.Lock(),
        "worker": None,
        "stop_event": threading.Event(),
        "ui": None,
    }
_guard = builtins._llm_judge_execution_guard
experiment_output = widgets.Output(layout=widgets.Layout(width="100%"))

def _other_workers_alive():
    return [
        t for t in threading.enumerate()
        if t.name == "judge-experiment-worker" and t.is_alive()
    ]

def _sync_busy_status():
    busy = bool(_other_workers_alive())
    experiment_ui.execution_busy = busy
    if busy and not experiment_ui.run_active:
        experiment_ui.run_status = (
            "Another experiment worker is active in this Kernel. "
            "Wait for it to finish before starting a new run."
        )
    return busy

def _request_stop(change):
    if _guard.get("ui") is experiment_ui and experiment_ui.run_active:
        _guard["stop_event"].set()
        experiment_ui.run_status = "Stopping after current run (CSV will be saved first)…"

def _execute_in_background(plan, ui, stop_event):
    try:
        def _report_progress(done, total, item):
            ui.run_total = total
            ui.run_current = done
            if stop_event.is_set():
                ui.run_status = f"Stopping — {done}/{total} completed"
            elif done < total:
                ui.run_status = f"Running — {done}/{total} completed"
            else:
                ui.run_status = f"Finishing — {done}/{total} completed"

        def _on_run_start(index, total, item):
            ui.current_run_started_at = time.time()
            ui.sample_current = 0
            ui.sample_total = 0
            ui.sample_last_completed_at = 0.0
            variant = (f" · {item['dynamic_prediction_variant']}"
                       if item["method"] == "dynamic" else "")
            ui.current_run_label = (
                f"Experiment {index}/{total}: {item['model']} · {item['dataset_id']} · "
                f"{item['method']} ({item['baseline_criterion_detail']}){variant}"
            )
            ui.run_status = f"Running experiment {index}/{total} — waiting for model response"

        def _on_sample_progress(done, total):
            ui.sample_current = done
            if total is not None:
                ui.sample_total = total
            if done:
                ui.sample_last_completed_at = time.time()

        def _on_run_result(record):
            ui.run_results = [*ui.run_results, record]

        with experiment_output:
            completed = execute_plan(
                plan, progress_callback=_report_progress,
                stop_requested=stop_event.is_set,
                on_run_start=_on_run_start,
                on_run_result=_on_run_result,
                on_sample_progress=_on_sample_progress,
            )
            if stop_event.is_set():
                ui.run_status = f"Stopped — {len(completed)}/{len(plan)} runs completed"
                print(f"Stopped. Completed {len(completed)}/{len(plan)} runs.")
            else:
                ui.run_status = f"Completed {len(completed)}/{len(plan)} runs"
                print(f"Completed {len(completed)} runs.")
    except Exception as exc:
        ui.run_status = f"Failed: {type(exc).__name__}: {exc}"
        with experiment_output:
            import traceback
            traceback.print_exc()
    finally:
        ui.current_run_started_at = 0.0
        ui.run_active = False
        with _guard["lock"]:
            if _guard.get("ui") is ui:
                _guard["ui"] = None
                _guard["worker"] = None
        ui.execution_busy = False

def _launch_from_ui(change):
    # Guard against duplicated notifications and re-executed notebook cells.
    with _guard["lock"]:
        if _other_workers_alive() or _guard.get("ui") is not None:
            experiment_ui.run_status = "Run blocked: an experiment worker is already active."
            experiment_ui.execution_busy = True
            return
        _guard["ui"] = experiment_ui
        _guard["stop_event"] = threading.Event()
    try:
        plan = build_plan(experiment_ui)
        if not plan:
            raise ValueError("No experiments selected.")
        experiment_ui.run_results = []
        experiment_ui.sample_current = 0
        experiment_ui.sample_total = 0
        experiment_ui.sample_last_completed_at = 0.0
        experiment_ui.current_run_label = ""
        experiment_ui.current_run_started_at = 0.0
        experiment_ui.run_active = True
        experiment_ui.execution_busy = True
        experiment_ui.run_status = f"Starting {len(plan)} experiments…"
        experiment_ui.run_current = 0
        experiment_ui.run_total = len(plan)
        with experiment_output:
            clear_output(wait=True)
            print(f"Starting {len(plan)} experiment(s).")
            print("Stop will take effect after the current run finishes.")
        worker = threading.Thread(
            target=_execute_in_background,
            args=(plan, experiment_ui, _guard["stop_event"]),
            daemon=True, name="judge-experiment-worker",
        )
        _guard["worker"] = worker
        worker.start()
    except Exception as exc:
        experiment_ui.run_active = False
        experiment_ui.execution_busy = False
        experiment_ui.run_status = f"Could not start: {type(exc).__name__}: {exc}"
        with _guard["lock"]:
            if _guard.get("ui") is experiment_ui:
                _guard["ui"] = None
                _guard["worker"] = None

experiment_ui.observe(_launch_from_ui, names="launch_request")
experiment_ui.observe(_request_stop, names="stop_request")
_sync_busy_status()
display(widgets.VBox([experiment_ui, experiment_output]))
